# Turb-DETR — Underwater Plastic Detection (Colab Training)

End-to-end training pipeline using **RT-DETR** via Ultralytics on underwater trash datasets.

| Item | Detail |
|---|---|
| Model | RT-DETR-L (COCO pretrained) |
| Datasets | Trash-ICRA19, UFO-120, TrashCan, RUIE |
| Framework | Ultralytics 8.3+ / PyTorch 2.2+ |
| Runtime | GPU (T4 or better) |

> **Before running:** Go to *Runtime → Change runtime type → **GPU*** 

## 1. Environment Setup

In [ ]:
# ── 1a. Check GPU availability ──────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# ── 1b. Set up repository ────────────────────────────────────
# OPTION A (default): Clone from GitHub.
#   Requires the repo to be pushed to GitHub first.
#   Push your local code: git push origin main
#
# OPTION B: Upload your repo folder to Google Drive, then:
#   import shutil
#   shutil.copytree("/content/drive/MyDrive/turb-detr-underwater-detection", REPO_DIR)
#
# Currently using OPTION A. If clone fails, switch to OPTION B.

import os

REPO_URL = "https://github.com/csdeepak/turb-detr-underwater-detection.git"
REPO_DIR = "/content/turb-detr-underwater-detection"

if not os.path.exists(REPO_DIR):
    result = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if result != 0:
        print("=" * 60)
        print("⚠  git clone FAILED.")
        print("   The GitHub repo may not be public or doesn't exist yet.")
        print("   OPTION B: upload the project folder to MyDrive and run:")
        print("     import shutil")
        print("     shutil.copytree(")
        print('       "/content/drive/MyDrive/turb-detr-underwater-detection",')
        print(f'       "{REPO_DIR}")')
        print("=" * 60)
    else:
        print(f"✓ Repository cloned to {REPO_DIR}")
else:
    print(f"✓ Repository already exists at {REPO_DIR}")

%cd {REPO_DIR}

In [ ]:
# ── 1c. Install project dependencies ────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 1d. Verify installation & GPU ───────────────────────────
import torch
import ultralytics

ultralytics.checks()  # prints ultralytics + system info

print(f"\nPyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"Device   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Ultralytics : {ultralytics.__version__}")

## 2. Datasets

**How this works:**
- Drive is mounted first (cell 2a)
- Each dataset cell will pause and ask you to paste the Google Drive URL
- Paste the URL and press Enter — it downloads and restructures automatically

In [ ]:
# ── 2a. Mount Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── 2b. Install gdown ────────────────────────────────────────
!pip install -q gdown
print("gdown ready.")

In [ ]:
# ── 2c. Restructure Trash-ICRA19 ────────────────────────────
# Strategy (tried in order):
#   1. Auto-detect dataset in MyDrive  (fastest — works if you own the folder)
#   2. Auto-detect in Shared Drives    (org/team drives)
#   3. Drive API download              (works for "Shared with me" folders —
#                                       uses your Colab OAuth session, full access)
#
# gdown is NOT used — it cannot access files that require Google login.

import shutil, os, re, io
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
raw_dir    = Path("data/_trash_icra19_raw")

# Pre-mapped folder IDs (from previous directory traversal)
_DATASET_FOLDER_ID = "1JHD3U4qS00iMAX52WrkFXyM5QSku88n6"  # trash_ICRA19/dataset/

# ── Step 1: Try to find dataset in mounted Drive paths ────────
def _find_in_mounted(base: Path):
    candidates = [
        base / "MyDrive" / "underwater_datasets" / "trash_ICRA19" / "dataset",
        base / "MyDrive" / "underwater_datasets" / "trash_ICRA19",
        base / "MyDrive" / "trash_ICRA19" / "dataset",
        base / "MyDrive" / "trash_ICRA19",
    ]
    for c in candidates:
        if not c.exists():
            continue
        sub = {p.name.lower() for p in c.iterdir() if p.is_dir()}
        if "train" in sub or "test" in sub:
            return c

    # Scan Shared Drives (org/team drives)
    shared = base / "Shareddrives"
    if shared.exists():
        for drive_dir in shared.iterdir():
            for root, dirs, _ in os.walk(str(drive_dir)):
                rp = Path(root)
                sub = {d.lower() for d in dirs}
                if "train" in sub and ("val" in sub or "test" in sub):
                    return rp
    return None

dataset_root = _find_in_mounted(DRIVE_ROOT)

if dataset_root:
    print(f"✓ Found dataset in mounted Drive: {dataset_root}")
else:
    # ── Step 2: Drive API download (Shared with me) ───────────
    # Uses the Colab OAuth session — works for any folder shared with your account.
    print("Dataset not found in mounted paths (likely in 'Shared with me').")
    print("Using authenticated Drive API to download …\n")

    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    import google.auth

    auth.authenticate_user()
    creds, _ = google.auth.default()
    service  = build("drive", "v3", credentials=creds)

    def _api_download_folder(svc, folder_id: str, dest: Path, _depth=0):
        """Recursively download a Drive folder via the authenticated API."""
        dest.mkdir(parents=True, exist_ok=True)
        page_token = None
        while True:
            resp = svc.files().list(
                q=f"'{folder_id}' in parents and trashed=false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()
            for f in resp.get("files", []):
                fpath = dest / f["name"]
                if f["mimeType"] == "application/vnd.google-apps.folder":
                    _api_download_folder(svc, f["id"], fpath, _depth + 1)
                else:
                    if fpath.exists():          # skip already-downloaded files
                        continue
                    try:
                        req = svc.files().get_media(
                            fileId=f["id"], supportsAllDrives=True
                        )
                        with open(fpath, "wb") as fh:
                            dl = MediaIoBaseDownload(fh, req)
                            done = False
                            while not done:
                                _, done = dl.next_chunk()
                        indent = "  " * _depth
                        print(f"{indent}↓ {f['name']}")
                    except Exception as e:
                        print(f"  [SKIP] {f['name']}: {e}")
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

    raw_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading trash_ICRA19/dataset/ → {raw_dir}")
    print("(This will take a few minutes for the full dataset)\n")
    _api_download_folder(service, _DATASET_FOLDER_ID, raw_dir)

    # Find the split root inside what was downloaded
    for c in [raw_dir, *sorted(raw_dir.rglob("*"))]:
        if not c.is_dir():
            continue
        sub = {p.name.lower() for p in c.iterdir() if p.is_dir()}
        if "train" in sub and ("val" in sub or "test" in sub):
            dataset_root = c
            break

    if dataset_root is None:
        print("\n✗ Download complete but train/val/test structure not found.")
        print("  Try: right-click the folder in Drive → Add shortcut to Drive,")
        print("  then re-run this cell (auto-detect will find it in MyDrive).")
        raise RuntimeError("Dataset not found after download attempt.")

    print(f"\n✓ Downloaded to: {dataset_root}")

# ── Step 3: Show what's in the source folder ─────────────────
print("\nSource contents:")
for split in ("train", "val", "test"):
    d = dataset_root / split
    if d.exists():
        imgs = sum(1 for f in d.iterdir() if f.suffix.lower() in {".jpg",".jpeg",".png"})
        txts = sum(1 for f in d.iterdir() if f.suffix == ".txt")
        print(f"  {split:5s}: {imgs} images, {txts} labels")
    else:
        print(f"  {split:5s}: (not found)")

# ── Step 4: Copy to YOLO layout ──────────────────────────────
# .jpg/.png → data/trash_icra19/images/<split>/
# .txt      → data/trash_icra19/labels/<split>/
# .xml (Pascal VOC) skipped — not used by Ultralytics
out_root   = Path("data/trash_icra19")
IMG_EXTS   = {".jpg", ".jpeg", ".png", ".bmp"}
SKIP_STEMS = {"readme", "classes"}

copied_any = False
for split_dir in sorted(dataset_root.iterdir()):
    if not split_dir.is_dir():
        continue
    split = split_dir.name.lower()
    if split not in ("train", "val", "test"):
        continue

    imgs_dst = out_root / "images" / split
    lbls_dst = out_root / "labels" / split
    imgs_dst.mkdir(parents=True, exist_ok=True)
    lbls_dst.mkdir(parents=True, exist_ok=True)

    n_i = n_l = 0
    for f in split_dir.rglob("*"):
        if not f.is_file():
            continue
        if f.suffix.lower() in IMG_EXTS:
            shutil.copy2(f, imgs_dst / f.name); n_i += 1
        elif f.suffix.lower() == ".txt" and f.stem.lower() not in SKIP_STEMS:
            shutil.copy2(f, lbls_dst / f.name); n_l += 1

    print(f"  {split:6s} → {n_i:>5} images  |  {n_l:>5} labels")
    copied_any = True

if not copied_any:
    raise RuntimeError("Nothing copied — check that the source has train/val/test folders.")

print(f"\n✓ Trash-ICRA19 ready at {out_root}")
print("→ Run cell 3a to verify counts, then cell 2d for TrashCAN.")


In [ ]:
# ── 2d. Download & restructure TrashCAN 1.0 ─────────────────
# Strategy (same as 2c):
#   1. Auto-detect in MyDrive / Shared Drives (instant if shortcut exists)
#   2. Drive API download via folder URL    (works for "Shared with me")
#
# TrashCAN ships COCO JSON — this cell converts to YOLO .txt after download.
# Drive structure:  TrashCAN 1.0/instance_version/{images/, *.json}

import shutil, os, json, re, io
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")

# ── Step 1: Auto-detect in mounted Drive ─────────────────────
def _find_trashcan(base: Path):
    candidates = [
        base / "MyDrive" / "underwater_datasets" / "TrashCAN 1.0" / "instance_version",
        base / "MyDrive" / "underwater_datasets" / "TrashCAN 1.0",
        base / "MyDrive" / "TrashCAN 1.0" / "instance_version",
        base / "MyDrive" / "TrashCAN 1.0",
        base / "MyDrive" / "TrashCAN" / "instance_version",
        base / "MyDrive" / "TrashCAN",
    ]
    for c in candidates:
        if c.exists():
            return c
    shared = base / "Shareddrives"
    if shared.exists():
        for drv in shared.iterdir():
            for root, dirs, _ in os.walk(str(drv)):
                rp = Path(root)
                files = [f.lower() for f in os.listdir(root)]
                if any("trashcan" in f and f.endswith(".json") for f in files):
                    return rp
    return None

tc_root = _find_trashcan(DRIVE_ROOT)

if tc_root:
    print(f"✓ Found TrashCAN in mounted Drive: {tc_root}")
    tc_raw = tc_root
else:
    # ── Step 2: Drive API download ────────────────────────────
    url = input("Paste Google Drive URL for TrashCAN 1.0 folder (Enter to skip): ").strip()
    if not url:
        print("Skipped — TrashCAN not downloaded.")
    else:
        m = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
        if not m:
            raise ValueError(f"Cannot extract folder ID from: {url!r}")
        folder_id = m.group(1)

        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        import google.auth

        auth.authenticate_user()
        creds, _  = google.auth.default()
        service   = build("drive", "v3", credentials=creds)

        def _api_dl(svc, fid: str, dest: Path, _d=0):
            dest.mkdir(parents=True, exist_ok=True)
            page_token = None
            while True:
                resp = svc.files().list(
                    q=f"'{fid}' in parents and trashed=false",
                    fields="nextPageToken, files(id, name, mimeType)",
                    pageToken=page_token,
                    supportsAllDrives=True, includeItemsFromAllDrives=True,
                ).execute()
                for f in resp.get("files", []):
                    fp = dest / f["name"]
                    if f["mimeType"] == "application/vnd.google-apps.folder":
                        _api_dl(svc, f["id"], fp, _d + 1)
                    elif not fp.exists():
                        try:
                            req = svc.files().get_media(fileId=f["id"], supportsAllDrives=True)
                            with open(fp, "wb") as fh:
                                dl = MediaIoBaseDownload(fh, req)
                                done = False
                                while not done: _, done = dl.next_chunk()
                            print("  " * _d + f"↓ {f['name']}")
                        except Exception as e:
                            print(f"  [SKIP] {f['name']}: {e}")
                page_token = resp.get("nextPageToken")
                if not page_token:
                    break

        tc_raw_root = Path("data/_trashcan_raw")
        tc_raw_root.mkdir(parents=True, exist_ok=True)
        print(f"Downloading TrashCAN 1.0 → {tc_raw_root}\n")
        _api_dl(service, folder_id, tc_raw_root)
        print("✓ Download complete.")

        # Find instance_version inside what we downloaded
        iv = next(tc_raw_root.rglob("instance_version"), tc_raw_root)
        tc_raw = iv

# ── Step 3: COCO JSON → YOLO .txt conversion ─────────────────
def coco_to_yolo(json_path, img_src, imgs_dst, lbls_dst):
    imgs_dst.mkdir(parents=True, exist_ok=True)
    lbls_dst.mkdir(parents=True, exist_ok=True)
    data    = json.loads(Path(json_path).read_text())
    id2img  = {img["id"]: img for img in data["images"]}
    ann_map = {}
    for ann in data.get("annotations", []):
        ann_map.setdefault(ann["image_id"], []).append(ann)
    ni = nl = 0
    for img_id, info in id2img.items():
        fname = Path(info["file_name"]).name
        src   = Path(img_src) / fname
        if src.exists():
            shutil.copy2(src, Path(imgs_dst) / fname);  ni += 1
        W, H  = info["width"], info["height"]
        lines = []
        for ann in ann_map.get(img_id, []):
            x, y, w, h = ann["bbox"]
            cx = (x + w/2)/W;  cy = (y + h/2)/H;  nw = w/W;  nh = h/H
            cls = ann["category_id"] - 1
            lines.append(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        if lines:
            (Path(lbls_dst) / (Path(fname).stem + ".txt")).write_text("\n".join(lines))
            nl += 1
    return ni, nl

# Find the folder containing the JSON files
iv_dir = None
for candidate in [tc_raw, *sorted(tc_raw.rglob("*"))]:
    if not candidate.is_dir():
        continue
    jsons = list(candidate.glob("*.json"))
    if any("trashcan" in j.name.lower() for j in jsons):
        iv_dir = candidate
        break

if iv_dir is None:
    print("[WARN] No TrashCAN JSON files found. Check the downloaded/mounted structure.")
else:
    out_tc = Path("data/trashcan")
    for split, jname in [("train", "instances_train_trashcan.json"),
                          ("val",   "instances_val_trashcan.json")]:
        jp = iv_dir / jname
        if not jp.exists():
            print(f"  [WARN] {jname} missing — skipping {split}");  continue
        ni, nl = coco_to_yolo(jp, iv_dir / "images",
                              out_tc / "images" / split,
                              out_tc / "labels" / split)
        print(f"  trashcan {split:5s} → {ni:>5} images | {nl:>5} labels")
    print(f"\n✓ TrashCAN ready at {out_tc}")
    print("→ Continue to cell 2e for UFO-120 (optional).")


In [ ]:
# ── 2e. UFO-120 (optional) ───────────────────────────────────
# UFO-120 is a super-resolution / enhancement benchmark — NO detection labels.
# Only needed for turbidity robustness evaluation (cell 5+), NOT for training.
# Press Enter to skip if you only need to train.
#
# Strategy (same as 2c/2d):
#   1. Auto-detect in MyDrive / Shared Drives
#   2. Drive API download via folder URL

import os, re
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")

# ── Step 1: Auto-detect in mounted Drive ─────────────────────
def _find_ufo(base: Path):
    candidates = [
        base / "MyDrive" / "underwater_datasets" / "UFO-120",
        base / "MyDrive" / "UFO-120",
        base / "MyDrive" / "UFO120",
    ]
    for c in candidates:
        if c.exists():
            return c
    shared = base / "Shareddrives"
    if shared.exists():
        for drv in shared.iterdir():
            for root, dirs, _ in os.walk(str(drv)):
                for d in dirs:
                    if "ufo" in d.lower() and "120" in d:
                        return Path(root) / d
    return None

ufo_root = _find_ufo(DRIVE_ROOT)

if ufo_root:
    print(f"✓ Found UFO-120 in mounted Drive: {ufo_root}")
    import shutil
    out = Path("data/ufo120")
    out.mkdir(parents=True, exist_ok=True)
    total = 0
    for f in ufo_root.rglob("*"):
        if f.is_file():
            dst = out / f.relative_to(ufo_root)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst);  total += 1
    print(f"  Copied {total} files → {out}")
else:
    url = input("\nPaste Google Drive URL for UFO-120 folder (Enter to skip): ").strip()
    if not url:
        print("Skipped — UFO-120 not needed for training.")
    else:
        m = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
        if not m:
            raise ValueError(f"Cannot extract folder ID from: {url!r}")
        folder_id = m.group(1)

        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        import google.auth

        auth.authenticate_user()
        creds, _ = google.auth.default()
        service  = build("drive", "v3", credentials=creds)

        def _api_dl(svc, fid: str, dest: Path, _d=0):
            dest.mkdir(parents=True, exist_ok=True)
            page_token = None
            while True:
                resp = svc.files().list(
                    q=f"'{fid}' in parents and trashed=false",
                    fields="nextPageToken, files(id, name, mimeType)",
                    pageToken=page_token,
                    supportsAllDrives=True, includeItemsFromAllDrives=True,
                ).execute()
                for f in resp.get("files", []):
                    fp = dest / f["name"]
                    if f["mimeType"] == "application/vnd.google-apps.folder":
                        _api_dl(svc, f["id"], fp, _d + 1)
                    elif not fp.exists():
                        try:
                            req = svc.files().get_media(fileId=f["id"], supportsAllDrives=True)
                            with open(fp, "wb") as fh:
                                dl = MediaIoBaseDownload(fh, req)
                                done = False
                                while not done: _, done = dl.next_chunk()
                            print("  " * _d + f"↓ {f['name']}")
                        except Exception as e:
                            print(f"  [SKIP] {f['name']}: {e}")
                page_token = resp.get("nextPageToken")
                if not page_token:
                    break

        out = Path("data/ufo120")
        out.mkdir(parents=True, exist_ok=True)
        print(f"Downloading UFO-120 → {out}\n")
        _api_dl(service, folder_id, out)
        print(f"✓ UFO-120 downloaded to {out}")


## 3. Verify Dataset Structure

In [ ]:
# ── 3a. Verify expected folder structure per dataset ─────────
from pathlib import Path

EXPECTED_SPLITS = ["train", "val", "test"]

for ds_name in ["trash_icra19", "ufo120", "trashcan", "ruie"]:
    ds_root = Path("data") / ds_name
    print(f"\n{'─'*50}")
    print(f"📂 {ds_name}  →  {'EXISTS' if ds_root.exists() else '⚠ MISSING'}")
    if not ds_root.exists():
        continue

    # Check for images/ and labels/ subdirectories per split
    for split in EXPECTED_SPLITS:
        img_dir = ds_root / "images" / split
        lbl_dir = ds_root / "labels" / split
        img_ok = img_dir.exists()
        lbl_ok = lbl_dir.exists()
        n_imgs = len(list(img_dir.glob("*"))) if img_ok else 0
        n_lbls = len(list(lbl_dir.glob("*"))) if lbl_ok else 0
        status = "✓" if (img_ok and lbl_ok) else "✗"
        print(f"  {status} {split:6s} — images: {n_imgs:>5,}  labels: {n_lbls:>5,}")

In [ ]:
# ── 3b. Print dataset statistics ─────────────────────────────
from pathlib import Path
from collections import Counter

def dataset_stats(ds_path: Path, class_names: list[str] | None = None):
    """Print image counts and class distribution for a YOLO-format dataset."""
    print(f"\n{'='*55}")
    print(f"  Dataset: {ds_path.name}")
    print(f"{'='*55}")

    total_imgs, total_objects = 0, 0
    class_counts = Counter()

    for split in ["train", "val", "test"]:
        img_dir = ds_path / "images" / split
        lbl_dir = ds_path / "labels" / split
        n_imgs = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
        total_imgs += n_imgs

        if lbl_dir.exists():
            for lbl_file in lbl_dir.glob("*.txt"):
                for line in lbl_file.read_text().strip().splitlines():
                    parts = line.strip().split()
                    if parts:
                        class_counts[int(parts[0])] += 1
                        total_objects += 1

        print(f"  {split:6s}: {n_imgs:>6,} images")

    print(f"  {'─'*40}")
    print(f"  Total : {total_imgs:>6,} images, {total_objects:>7,} objects")

    if class_counts:
        print(f"\n  Class distribution:")
        for cls_id in sorted(class_counts):
            name = class_names[cls_id] if class_names and cls_id < len(class_names) else f"class_{cls_id}"
            pct = class_counts[cls_id] / total_objects * 100
            bar = "█" * int(pct / 2)
            print(f"    {cls_id}: {name:15s} {class_counts[cls_id]:>6,}  ({pct:5.1f}%) {bar}")

# Run stats for Trash-ICRA19 (primary dataset)
trash_icra19_classes = ["plastic", "bottle", "can", "bag", "net"]
ds = Path("data/trash_icra19")
if ds.exists():
    dataset_stats(ds, trash_icra19_classes)
else:
    print("⚠ data/trash_icra19 not found — check symlinks above.")

## 4. Train Models

**Run 4a (baseline) first**, then 4b (Turb-DETR).
Both must complete before evaluation and ablation are meaningful.

| Run | Model | SimAM | Purpose |
|---|---|---|---|
| 4a | RT-DETR (baseline) | ✗ | Comparison baseline |
| 4b | Turb-DETR | ✓ | Your architecture |

In [ ]:
# ── 4a. Train BASELINE RT-DETR (no SimAM) ───────────────────
# This is your controlled baseline for comparison.
# use_simam=False → vanilla RT-DETR, no architecture change.
import sys
sys.path.insert(0, "/content/turb-detr-underwater-detection")

from models.turb_detr import TurbDETR

baseline = TurbDETR(model_variant="rtdetr-l", use_simam=False)
baseline.info()

baseline_results = baseline.train(
    data_cfg="configs/trash_icra19.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.01,
    weight_decay=1e-4,
    warmup_epochs=3,
    patience=15,
    amp=True,
    workers=4,
    project="outputs",
    name="rtdetr_baseline",
    save_period=10,
    exist_ok=True,
    verbose=True,
)

print("\nBaseline training complete.")
print(f"mAP@0.5      : {baseline_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP@0.5:0.95 : {baseline_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

In [ ]:
# ── 4b. Train TURB-DETR (with SimAM) ────────────────────────
# Your architecture: RT-DETR + SimAM turbidity suppression.
# This is the model you are comparing against the baseline above.

from models.turb_detr import TurbDETR

turb_detr = TurbDETR(model_variant="rtdetr-l", use_simam=True)
turb_detr.info()

turb_results = turb_detr.train(
    data_cfg="configs/trash_icra19.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.01,
    weight_decay=1e-4,
    warmup_epochs=3,
    patience=15,
    amp=True,
    workers=4,
    project="outputs",
    name="turb_detr",
    save_period=10,
    exist_ok=True,
    verbose=True,
)

print("\nTurb-DETR training complete.")
print(f"mAP@0.5      : {turb_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP@0.5:0.95 : {turb_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

## 5. Evaluate & Visualize

In [ ]:
# ── 5a. Compare baseline vs Turb-DETR on test set ───────────
from pathlib import Path

BASELINE_WEIGHTS  = "outputs/rtdetr_baseline/weights/best.pt"
TURBDETR_WEIGHTS  = "outputs/turb_detr/weights/best.pt"

results_table = []

for name, weights, use_simam in [
    ("RT-DETR baseline",  BASELINE_WEIGHTS, False),
    ("Turb-DETR (SimAM)", TURBDETR_WEIGHTS, True),
]:
    wpath = Path(weights)
    if not wpath.exists():
        print(f"⚠ {name}: weights not found at {weights} — skip")
        continue

    from models.turb_detr import TurbDETR
    m = TurbDETR(weights=weights, use_simam=use_simam)
    metrics = m.validate(data_cfg="configs/trash_icra19.yaml", split="test")

    row = {
        "Model":     name,
        "mAP@0.5":   metrics.box.map50,
        "mAP@0.5:95":metrics.box.map,
        "Precision": metrics.box.mp,
        "Recall":    metrics.box.mr,
    }
    results_table.append(row)
    del m  # free GPU memory

print(f"\n{'Model':<24} {'mAP@0.5':>9} {'mAP@.5:.95':>12} {'Prec':>8} {'Recall':>8}")
print("─" * 64)
for r in results_table:
    print(f"{r['Model']:<24} {r['mAP@0.5']:>9.4f} {r['mAP@0.5:95']:>12.4f} {r['Precision']:>8.4f} {r['Recall']:>8.4f}")

# ── 5b. Inference on sample test images ──────────────────────
from IPython.display import Image, display
from pathlib import Path
import sys, glob

sys.path.insert(0, "/content/turb-detr-underwater-detection")
from models.turb_detr import TurbDETR

# Load best Turb-DETR weights for inference
TURBDETR_WEIGHTS = "outputs/turb_detr/weights/best.pt"
if not Path(TURBDETR_WEIGHTS).exists():
    print(f"⚠ {TURBDETR_WEIGHTS} not found — run training cells first.")
else:
    infer_model = TurbDETR(weights=TURBDETR_WEIGHTS, use_simam=True)

    test_dir     = Path("data/trash_icra19/images/test")
    sample_imgs  = sorted(glob.glob(str(test_dir / "*")))[:4]

    if not sample_imgs:
        print(f"⚠ No images found in {test_dir}")
    else:
        preds = infer_model.predict(
            source=sample_imgs,
            imgsz=640,
            conf=0.25,
            save=True,
            project="outputs",
            name="turb_detr_predictions",
            exist_ok=True,
        )
        pred_dir = Path("outputs/turb_detr_predictions")
        for img_path in sorted(pred_dir.glob("*.jpg"))[:4]:
            display(Image(filename=str(img_path), width=640))
        del infer_model

In [ ]:
# ── 5c. Save all weights to Google Drive ─────────────────────
import shutil
from pathlib import Path

dst = Path("/content/drive/MyDrive/turb_detr_weights/")
dst.mkdir(parents=True, exist_ok=True)

weights_to_save = {
    "rtdetr_baseline_best.pt":  "outputs/rtdetr_baseline/weights/best.pt",
    "rtdetr_baseline_last.pt":  "outputs/rtdetr_baseline/weights/last.pt",
    "turb_detr_best.pt":        "outputs/turb_detr/weights/best.pt",
    "turb_detr_last.pt":        "outputs/turb_detr/weights/last.pt",
}

for dst_name, src_path in weights_to_save.items():
    src = Path(src_path)
    if src.exists():
        shutil.copy2(src, dst / dst_name)
        print(f"  ✓ {dst_name:<35} {src.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"  ✗ {dst_name:<35} not found (training may not have completed)")

print(f"\nAll available weights saved to {dst}")